In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, MeanShift
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [4]:
wine_data = pd.read_csv('wine-red.csv')

In [5]:
X = wine_data.drop('quality', axis=1)
y = wine_data['quality']

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [7]:
models = {
    "KMeans": KMeans(n_clusters=3, random_state=42),
    "HAC": AgglomerativeClustering(n_clusters=3),
    "DBSCAN": DBSCAN(eps=0.5, min_samples=5),
    "Mean Shift": MeanShift(bandwidth=None)
}
clustering_results = {}
for name, model in models.items():

    model.fit(X_scaled)

    labels = model.labels_
    if len(set(labels)) > 1:
        silhouette = silhouette_score(X_scaled, labels)
    else:
        silhouette = "N/A" 
    
    clustering_results[name] = {
        "labels": labels,
        "silhouette_score": silhouette
    }

clustering_results["KMeans"]["silhouette_score"], clustering_results["HAC"]["silhouette_score"], clustering_results["DBSCAN"]["silhouette_score"], clustering_results["Mean Shift"]["silhouette_score"]

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


(0.1892040681108649,
 0.15774678821176108,
 -0.25051192044580745,
 0.327693236989421)

Các kết quả điểm Silhouette cho từng phương pháp phân cụm trên bộ dữ liệu rượu vang đỏ như sau:

KMeans: 0.189

HAC (Hierarchical Agglomerative Clustering): 0.158

DBSCAN: -0.251

Mean Shift: 0.328

Điểm Silhouette có giá trị từ -1 đến 1, nơi giá trị cao hơn chỉ ra rằng các điểm dữ liệu được phân bố tốt trong cụm của chúng và xa các cụm khác. 
Dựa trên kết quả này, có thể thấy:

Mean Shift có điểm Silhouette cao nhất, cho thấy kỹ thuật này có thể tạo ra các cụm có sự phân tách tốt nhất trong bộ dữ liệu này.

KMeans và HAC đều cho kết quả khá, với điểm Silhouette dương nhưng không cao, cho thấy các cụm có sự phân tách nhưng không quá rõ rệt.

DBSCAN có điểm Silhouette âm, chỉ ra rằng phương pháp này có thể không phù hợp với cấu trúc dữ liệu này, hoặc có thể cần điều chỉnh các tham số để đạt được kết quả tốt hơn.

In [8]:

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

kmeans_pca = KMeans(n_clusters=3, random_state=42)
kmeans_pca.fit(X_pca)

silhouette_score_pca = silhouette_score(X_pca, kmeans_pca.labels_)

X_pca.shape, silhouette_score_pca



c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


((1599, 2), 0.3778647807456886)

Sau khi áp dụng PCA để giảm chiều dữ liệu xuống còn 2 chiều và thực hiện phân cụm bằng k-means, chúng ta nhận được điểm Silhouette là 0.378. So với kết quả trước khi giảm chiều, điểm Silhouette này cao hơn, cho thấy việc giảm chiều có thể giúp cải thiện khả năng phân cụm của dữ liệu.


Giảm chiều dữ liệu bằng PCA không chỉ giúp làm giảm độ phức tạp của dữ liệu mà còn cải thiện khả năng phân biệt giữa các cụm, như được chứng minh bởi điểm Silhouette cao hơn sau khi giảm chiều.

PCA có thể loại bỏ "tiếng ồn" từ dữ liệu và tập trung vào những thông tin quan trọng nhất, điều này giúp k-means hoạt động hiệu quả hơn trong việc phân biệt các cụm.

Các kỹ thuật phân cụm khác nhau có thể phù hợp với các loại cấu trúc dữ liệu khác nhau. Trong trường hợp này, Mean Shift và k-means sau PCA cho thấy hiệu quả tốt, trong khi DBSCAN không phù hợp với cấu trúc dữ liệu cụ thể này.